# Range-Restriction Normalization — Demo

A rule is **range-restricted** when every variable in its head also occurs in a
*range-generating* subgoal of its body. Operationally, a range-restricted
program starts from ground facts and only ever asserts ground items — which is
what lets the fast bottom-up (ground) solver run and what McAllester's
meta-complexity bound assumes.

`RangeRestrictionNormalization` rewrites **any** program into an equivalent one
whose non-range-restriction is confined to an explicit *residual layer* at the
outputs — exactly the way ε-production elimination in a CFG pushes the one
unavoidable case into a single `S0 → S | ε` rule. The construction is
**rename + recovery**: an open position is projected to a fresh lower-arity
symbol `q\j`, and a *recovery rule* `q(..,V,..) += q\j` re-expresses the
original. The surviving non-range-restricted recovery rules **are** the residue.

> Full specification: [`range-restriction-normalization.md`](range-restriction-normalization.md).
> This notebook walks the spec's worked examples E1–E6, then discusses
> **limitations** (the last section — read it before relying on the transform).

In [1]:
from dyna import Program
from dyna.analyze.range_restriction import (
    open_types, is_range_restricted, is_rule_range_restricted, bindable_vars,
)
from IPython.display import display, Markdown
import warnings

def same_values(p, q, queries, D=''):
    "Assert source and normalized agree on every ground query; return a check mark."
    ps, qs = (p + D).sol(), (q + D).sol()
    for query in queries:
        qs.assert_equal_query(query, ps.user_query(query))
    return Markdown('✓ **identical ground-query values** on: ' + ', '.join(f'`{x}`' for x in queries))

def show_residue(q):
    "Display the residual layer as a program (clean HTML), or note it is empty."
    if q.residual_layer.rules:
        display(Markdown('**residual layer** — the confined non-range-restriction:'))
        display(q.residual_layer)
    else:
        display(Markdown('**residual layer:** _(empty)_'))

## The analysis underneath (Step A) — and why the syntactic check isn't enough

Openness is computed as the `$free`-marked derivable *types* of the program
(`open_types`). The transform's stop condition is a **refined**
range-restriction check that counts only *generating* occurrences: tests
(`<`, `>`, `==`, …) generate nothing, while the binder builtins `=`/`is`
propagate (the engine inverts `X is Y + 1`).

The purely syntactic `Program.is_range_restricted` (`head_vars ⊆ body_vars`)
gets this wrong — it accepts a rule whose head variable is pinned only by a
test:

In [2]:
p = Program('''
f(X) += g(Y) * (X > Y).      % X is constrained only by the test X > Y
goal += f(X).
inputs: g(Y).
outputs: goal.
''')
print('syntactic Program.is_range_restricted :', p.is_range_restricted(), '   <- wrong')
print('refined   is_range_restricted         :', is_range_restricted(p), '  <- right')

syntactic Program.is_range_restricted : True    <- wrong
refined   is_range_restricted         : False   <- right


## E1 — canonical projection + summation (residue = scalar ∞)

`f(X) += 3` makes `f`'s argument *only-open* (no derivation ever grounds it).
In `goal += f(X)` the variable `X` is summed over the whole universe, so the
projection leaves an explicit infinite-multiplicity factor. `goal` is ground,
so the **residue is empty**.

In [3]:
p = Program('''
f(X) += 3.
goal += f(X).
outputs: goal.
''')
q = p.normalize_range_restriction()
display(Markdown('**source**'));     display(p)
display(Markdown('**normalized**')); display(q)
show_residue(q)

**source**

{
  0: f(X) += 3.
  1: goal += f(X).
}

**normalized**

{
  0: goal += goal_1.
  1: f_0 += 3.
  2: goal_1 += f_0 * inf.
}

**residual layer:** _(empty)_

In [4]:
# the ∞ is produced via Semiring.multiple, not hard-coded — value preserved
p.sol().assert_equal_query('goal', float('inf'))
q.sol().assert_equal_query('goal', float('inf'))
display(same_values(p, q, ['goal']))

✓ **identical ground-query values** on: `goal`

## E2 — recursion through a threaded open variable (no infinite unfolding)

Here the open variable `X0` is *threaded* — preserved on both head and body —
not summed. So **no** multiplicity factor is added, and the reflexive base rule
`temp(X0,X0) += 1` is the irreducible residue. The expansion is one-shot;
recursion does not cause a loop.

In [5]:
p = Program('''
temp(X0, X0) += 1.
temp(X, X0)  += rewrite(X, Y) * temp(Y, X0).
inputs: rewrite(X, Y).
outputs: temp(X, X0).
''')
q = p.normalize_range_restriction()
display(q)
show_residue(q)

{
  0: rewrite_0(X,Y) += rewrite(X,Y).
  1: temp(X,X0) += temp_1(X,X0).
  2: temp(X0,X0) += temp_2.
  3: temp_2 += 1.
  4: temp_1(X,X0) += rewrite_0(X,Y) * temp_1(Y,X0).
  5: temp_1(X,X0) += rewrite_0(X,X0) * temp_2.
}

**residual layer** — the confined non-range-restriction:

{
  0: temp(X0,X0) += temp_2.
}

In [6]:
# the diagonal is the value-bearing part: in-place elimination would DROP it.
# (With temp declared an output, elim(0) is correctly *refused*; on a copy
# without the declaration it overwrites temp and loses the reflexive +1.)
display(Markdown('`elim(0)` is the lossy in-place alternative — on an output-free '
                 'copy it drops the reflexive diagonal (`temp(a,a)` loses its `+1`):'))
p_noout = Program('''
temp(X0, X0) += 1.
temp(X, X0)  += rewrite(X, Y) * temp(Y, X0).
''')
display(p_noout.elim(0))

`elim(0)` is the lossy in-place alternative — on an output-free copy it drops the reflexive diagonal (`temp(a,a)` loses its `+1`):

{
  0: temp(X,X0) += rewrite(X,X0).
  1: temp(X,X0) += rewrite(X,Y) * temp(Y,X0).
}

In [7]:
D = 'rewrite(a,b) += 0.5. rewrite(b,c) += 0.25.'
display(same_values(p, q, ['temp(a,c)', 'temp(a,a)', 'temp(b,b)', 'temp(a,b)'], D=D))

✓ **identical ground-query values** on: `temp(a,c)`, `temp(a,a)`, `temp(b,b)`, `temp(a,b)`

## E3 — an open variable reaches an output (residue = one open rule)

Now the open variable flows to a *declared output*, so it cannot be summed
away. Step D isolates the single surviving open rule into `residual_layer`;
everything beneath it is range-restricted. This is the `S0 → ε` analog.

In [8]:
p = Program('''
f(X) += 3.
out(X) += f(X).      % out is an output; X reaches it open
outputs: out(X).
''')
q = p.normalize_range_restriction()
display(q)
print('engine layer range-restricted? ', is_range_restricted(q.engine_layer))
show_residue(q)
display(same_values(p, q, ['out(z)']))

{
  0: out(X) += out_1.
  1: f_0 += 3.
  2: out_1 += f_0.
}

engine layer range-restricted?  True


**residual layer** — the confined non-range-restriction:

{
  0: out(X) += out_1.
}

✓ **identical ground-query values** on: `out(z)`

## E4 — active-domain escape (residue removed)

Supplying an active-domain predicate `adom` (`Step E`) splices `adom(X)` into
the open readings. The residue **disappears** — the whole program becomes
range-restricted — at the cost of materializing the domain. Every `∞` witness
count becomes `|adom|`.

In [9]:
p = Program('''
f(X) += 3.
out(X) += f(X).
outputs: out(X).
''')
q = p.normalize_range_restriction(adom='adom')
display(q)
show_residue(q)
print('fully range-restricted? ', is_range_restricted(q))

# out(z) = 3 for each z in the supplied domain
D = 'adom(a) += 1.0. adom(b) += 1.0.'
qs = (q + D).sol()
qs.assert_equal_query('out(a)', 3.0)
qs.assert_equal_query('out(b)', 3.0)
display(Markdown('✓ `out(a) = out(b) = 3` under the active domain `{a, b}`'))

{
  0: out(X) += out_1 * adom(X).
  1: f_0 += 3.
  2: out_1 += f_0.
}

**residual layer:** _(empty)_

fully range-restricted?  True


✓ `out(a) = out(b) = 3` under the active domain `{a, b}`

## E5 — a test-only variable is **irreducible**, not projectable

This is the subtle one, and it is a genuine **limitation** of what the
transform can normalize away without an active domain.

`X` in `f(X) += g(Y) * (X > Y)` is non-range-restricted, but it is **not**
pass-through: the value of `f(X)` genuinely depends on `X` (with `g(3)=1`,
`f(5)=1` but `f(2)=0`). Projecting it would assign every `f(_)` the same value
— corrupting ground queries. So the transform **leaves the rule untouched** as
residue of the *delayed-test* kind. (An earlier draft of the spec said to treat
this like E1; the implementation proved that value-unsound, and the spec was
corrected.)

In [10]:
p = Program('''
f(X) += g(Y) * (X > Y).
goal += f(X).
inputs: g(Y).
outputs: goal.
''')
q = p.normalize_range_restriction()
display(q)
display(Markdown('**residue** (delayed-test kind — the rule survives verbatim):'))
display(q.residual_layer)
print('engine layer range-restricted? ', is_range_restricted(q.engine_layer))

{
  0: f(X) += g(Y) * (X > Y).
  1: goal += f(X).
}

**residue** (delayed-test kind — the rule survives verbatim):

{
  0: f(X) += g(Y) * (X > Y).
}

engine layer range-restricted?  True


In [11]:
# with an active domain it becomes range-restricted like everything else:
q_adom = p.normalize_range_restriction(adom='adom')
display(q_adom)
D = 'g(3) += 1.0. adom(2) += 1.0. adom(5) += 1.0.'
qs = (q_adom + D).sol()
qs.assert_equal_query('f(5)', 1.0)   # 5 > 3
qs.assert_equal_query('f(2)', 0.0)   # 2 > 3 is false -- value DOES depend on X
display(Markdown('✓ `f(5) = 1`, `f(2) = 0` — the dependence on `X` that projection would have destroyed'))

{
  0: f(X) += g(Y) * (X > Y) * adom(X).
  1: goal += f(X).
}

✓ `f(5) = 1`, `f(2) = 0` — the dependence on `X` that projection would have destroyed

## E6 — nested sharing (the *no-flatness* canary)

The transform never keys on a top-level argument index. E6 folds E2's diagonal
*inside* one argument: `temp(pair(X0, X0))`. A flat, position-based analysis
would see one opaque open slot and lose the `X0 = X0` equality. The
type-pattern substrate keeps the sharing, so E6 normalizes **identically** to
E2.

In [12]:
p = Program('''
temp(pair(X0, X0)) += 1.
temp(pair(X, X0))  += rewrite(X, Y) * temp(pair(Y, X0)).
inputs: rewrite(X, Y).
outputs: temp(W).
''')
q = p.normalize_range_restriction()
display(q)
display(Markdown('**residue keeps the shared diagonal:**'))
display(q.residual_layer)
display(same_values(p, q, ['temp(pair(a,b))', 'temp(pair(a,a))', 'temp(pair(b,b))'],
                    D='rewrite(a,b) += 0.5.'))

{
  0: rewrite_0(X,Y) += rewrite(X,Y).
  1: temp(pair(X,X0)) += temp_1(X,X0).
  2: temp(pair(X0,X0)) += temp_2.
  3: temp_2 += 1.
  4: temp_1(X,X0) += rewrite_0(X,Y) * temp_1(Y,X0).
  5: temp_1(X,X0) += rewrite_0(X,X0) * temp_2.
}

**residue keeps the shared diagonal:**

{
  0: temp(pair(X0,X0)) += temp_2.
}

✓ **identical ground-query values** on: `temp(pair(a,b))`, `temp(pair(a,a))`, `temp(pair(b,b))`

### Invariance (no flatness, restated as a test)

Because openness lives in term patterns rather than argument indices, the
verdicts are invariant under **functor renaming** and **dummy-functor
wrapping**. Below, E3 and a renamed-and-wrapped copy of it normalize to the
same shape and the same values.

In [13]:
base    = Program('f(X) += 3.\nout(X) += f(X).\noutputs: out(X).')
wrapped = Program('zap(w(X)) += 3.\nquux(X) += zap(w(X)).\noutputs: quux(X).')

for name, prog in [('base', base), ('renamed+wrapped', wrapped)]:
    q = prog.normalize_range_restriction()
    print(f'{name:16s} residue={len(q.residual_layer.rules)}  engineRR={is_range_restricted(q.engine_layer)}')
    prog.sol().assert_equal_query(prog.outputs.rules[0].head.fn + '(z)' if False else
                                  ('out(z)' if name=='base' else 'quux(z)'),
                                  3.0)
    q.sol().assert_equal_query('out(z)' if name=='base' else 'quux(z)', 3.0)
display(Markdown('✓ identical structure and values under renaming + wrapping'))

base             residue=1  engineRR=True
renamed+wrapped  residue=1  engineRR=True


✓ identical structure and values under renaming + wrapping

## Post-condition checks (silent-failure guards)

Two detectable failure modes warn (both are *confinement* conditions — ground
values stay correct regardless):

1. **Non-convergence** — the projection loop hit `max_passes` without a clean
   break (`q.converged`).
2. **Residue shape** — every residual rule must be recovery-form or
   delayed-test-form; anything else means the loop gave up rather than that the
   residue is irreducible.

A clean run sets `converged = True` and emits nothing:

In [14]:
q = Program('f(X) += 3.\nout(X) += f(X).\noutputs: out(X).').normalize_range_restriction()
print('converged:', q.converged)

converged: True


In [15]:
# starving the loop (max_passes=0) trips the warning:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    q = Program('f(X) += 3.\nout(X) += f(X).\noutputs: out(X).').normalize_range_restriction(max_passes=0)
print('converged:', q.converged)
for w in caught:
    print('⚠', str(w.message)[:78])

converged: False
⚠ RangeRestrictionNormalization did not converge in 0 passes; the result may be 
⚠ RangeRestrictionNormalization residual layer contains 1 rule(s) that are neith


## Limitations

The transform is sound (ground-query values are always preserved), but it is
not omnipotent. Know these before relying on it:

1. **The residue is irreducible without an active domain.** When the universe
   is infinite and the program is domain-independent, the residual layer is
   *non-empty by necessity* — the analog of "you cannot express `ε ∈ L(G)`
   with zero ε-rules." This is the intended caveat (E1/E3), not a defect.
   `adom` (Step E) removes it, but commits to active-domain semantics and pays
   to materialize the domain.

2. **Test-only variables cannot be projected (E5).** A head variable pinned
   only by a test is non-range-restricted *and* not pass-through, so it stays
   as delayed-test residue. Without `adom` the engine must run it on the
   non-ground solver. This is a hard limit on what "normalization" can buy you,
   not a bug.

3. **Open inputs must be *declared*.** An input that arrives with a free
   variable is open by declaration, but the transform has no way to detect this
   — it assumes declared inputs arrive ground. You must pass it explicitly via
   `input_type=` (e.g. `Program('h(X) += $free(X).')`). If you don't, the
   transform silently under-normalizes (values still correct, but the openness
   is left for the runtime solver). There is no warning for this case because
   there is no signal to trigger one.

4. **Binder builtins are assumed invertible.** The refined check treats `=` and
   `is` as fully invertible (matching the engine's inversion of e.g.
   `X is Y + 1`). A builtin chain that is *not* invertible could be classified
   range-restricted yet raise `InstFault` at solve time. This is a
   *misclassification* risk — never a wrong value.

5. **One `∞` factor covers multiple vanished summed variables.** This is
   Abbreviate's convention and is exact in every semiring shipped here. A
   semiring where `multiple(∞) ⊗ multiple(∞) ≠ multiple(∞)` would need the
   per-variable product (which `adom` mode already does).

6. **`input_type` seeds the first pass only.** It describes the *original*
   inputs; later passes operate on the abbreviated program. A hypothetical
   multi-pass scenario that needed open inputs re-seeded on a later pass would
   miss them. In practice one projection pass normalizes the engine layer, so
   this is a sharp edge rather than an active problem.

7. **`is_input`/`is_output` are approximate.** Output/residue membership is
   decided by head-unification and does **not** inspect delayed guards on a
   declared head (`program.py`, marked `XXX: approximate`). Adequate for these
   examples; route through the `Program` query methods if a declared output
   head carries a guard the residue test must respect.

Let's make limitation **1** concrete — without `adom` the residue is forced,
with it the program is fully range-restricted:

In [16]:
p = Program('f(X) += 3.\nout(X) += f(X).\noutputs: out(X).')
print('without adom — residue rules:', len(p.normalize_range_restriction().residual_layer.rules))
print('with    adom — residue rules:', len(p.normalize_range_restriction(adom='adom').residual_layer.rules))

without adom — residue rules: 1
with    adom — residue rules: 0


## Summary

| Example | Open variable | Residue | Notes |
|---|---|---|---|
| E1 | summed, only-open | empty (∞ factor) | reproduces `test_infinite_multiplicity` |
| E2 | threaded through recursion | reflexive diagonal | no ∞ factor; one-shot, no unfolding |
| E3 | reaches an output | one open output rule | the `S0 → ε` analog |
| E4 | E3 + `adom` | empty | database "safe-ification" |
| E5 | test-only | delayed-test rule | **not** pass-through; irreducible without `adom` |
| E6 | shared, nested in `pair(·,·)` | shared diagonal | no-flatness canary; identical to E2 |

The engine layer is range-restricted under the refined check; the residue is
confined, classified, and (with `adom`) removable. See the **Limitations**
section above for what the transform cannot do.